# AVISO processed eddy dataset: quick analysis

This notebook opens the final processed Parquet table and provides simple quality-control summaries and plots. It does not modify the dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path(
    "/srv/scratch/z5297792/aviso_eddy_dataset/processed/eddy_dataset_processed.parquet"
)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Eddy", "Day"]).reset_index(drop=True)
print(f"Loaded {len(df):,} eddy-day rows from {DATA_PATH}")
df.head()

## Dataset overview

In [ ]:
overview = pd.Series(
    {
        "eddy-day rows": len(df),
        "unique eddies": df["Eddy"].nunique(),
        "first date": df["Date"].min(),
        "last date": df["Date"].max(),
        "cyclonic eddies": df.loc[df["Cyc"].eq("CE"), "Eddy"].nunique(),
        "anticyclonic eddies": df.loc[df["Cyc"].eq("AE"), "Eddy"].nunique(),
        "median lifetime (days)": df.groupby("Eddy").size().median(),
        "median radius (km)": df["R"].median(),
    },
    name="value",
)
overview.to_frame()

In [ ]:
pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": 100 * df.isna().mean(),
        "unique_values": df.nunique(dropna=True),
    }
).sort_values("missing_percent", ascending=False)

## One row per eddy

In [ ]:
eddy_summary = (
    df.groupby("Eddy")
    .agg(
        Cyc=("Cyc", "first"),
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        observations=("Day", "size"),
        mean_radius_km=("R", "mean"),
        mean_omega=("Omega", "mean"),
        mean_lon=("lon", "mean"),
        mean_lat=("lat", "mean"),
    )
    .reset_index()
)
eddy_summary["duration_days"] = (
    eddy_summary["end_date"] - eddy_summary["start_date"]
).dt.days + 1
eddy_summary.sort_values("duration_days", ascending=False).head(20)

## Basic distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

eddy_summary.groupby("Cyc").size().reindex(["CE", "AE"]).plot.bar(
    ax=axes[0], color=["tab:blue", "tab:red"]
)
axes[0].set(title="Unique eddies by polarity", xlabel="Polarity", ylabel="Eddies")
axes[0].tick_params(axis="x", rotation=0)

eddy_summary["duration_days"].plot.hist(ax=axes[1], bins=30, color="0.35")
axes[1].set(title="Eddy lifetime", xlabel="Duration (days)", ylabel="Eddies")

for polarity, color in [("CE", "tab:blue"), ("AE", "tab:red")]:
    df.loc[df["Cyc"].eq(polarity), "R"].dropna().plot.hist(
        ax=axes[2], bins=30, alpha=0.5, label=polarity, color=color
    )
axes[2].set(title="Radius distribution", xlabel="R (km)", ylabel="Eddy-days")
axes[2].legend()

fig.tight_layout()

## Spatial coverage

Plot one mean position per tracked eddy to keep the figure lightweight.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for polarity, color in [("CE", "tab:blue"), ("AE", "tab:red")]:
    subset = eddy_summary.loc[eddy_summary["Cyc"].eq(polarity)]
    ax.scatter(
        subset["mean_lon"], subset["mean_lat"],
        s=12, alpha=0.55, color=color, label=f"{polarity} ({len(subset):,})"
    )
ax.set(
    title="Mean position of each tracked eddy",
    xlabel="Longitude (°E)",
    ylabel="Latitude (°N)",
)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

## Temporal coverage

In [ ]:
monthly = (
    df.set_index("Date")
    .groupby("Cyc")
    .resample("MS")
    .size()
    .unstack(0, fill_value=0)
)
ax = monthly.plot(figsize=(13, 4), color={"CE": "tab:blue", "AE": "tab:red"})
ax.set(title="Monthly eddy-day observations", xlabel="Date", ylabel="Eddy-days")
ax.grid(alpha=0.25)
plt.tight_layout()

## Inspect an individual eddy

Change `EDDY_ID` to inspect its track and fitted properties.

In [ ]:
EDDY_ID = int(eddy_summary.sort_values("duration_days", ascending=False).iloc[0]["Eddy"])
eddy = df.loc[df["Eddy"].eq(EDDY_ID)].sort_values("Date")
print(f"Eddy {EDDY_ID}: {eddy.Cyc.iloc[0]}, {len(eddy)} days")
eddy.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(eddy["lon"], eddy["lat"], marker=".", ms=3)
axes[0].scatter(eddy["lon"].iloc[0], eddy["lat"].iloc[0], color="limegreen", label="Start")
axes[0].scatter(eddy["lon"].iloc[-1], eddy["lat"].iloc[-1], color="black", label="End")
axes[0].set(title=f"Eddy {EDDY_ID} track", xlabel="Longitude (°E)", ylabel="Latitude (°N)")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(eddy["Date"], eddy["R"], label="R (km)")
axes[1].set(title=f"Eddy {EDDY_ID} radius", xlabel="Date", ylabel="Radius (km)")
axes[1].grid(alpha=0.25)
fig.tight_layout()